<!-- course-header -->
# Lesson 05: Neural networks from scratch

*Worked solutions for lesson 05. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/05-neural-networks.html) or [offline](../../lessons/05-neural-networks.html). The exercise version without solutions is `../05-neural-networks.ipynb`.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/solutions/05-neural-networks-solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Build a multilayer neural network in plain NumPy, derive and implement backpropagation yourself, then learn the practical tricks (initialisation, Adam, normalisation, dropout) that make deep networks train.

**Runs on:** Any computer (CPU).

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy scikit-learn

## The forward pass

In [ ]:
import numpy as np
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=0)
y = y.reshape(-1, 1).astype(float)          # (1000, 1) so shapes line up with the output
rng = np.random.default_rng(0)

def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))

def init(sizes):
    """He initialisation: weights ~ N(0, 2/n_in), good for ReLU."""
    params = {}
    for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:]), start=1):
        params[f"W{i}"] = rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out))
        params[f"b{i}"] = np.zeros(n_out)
    return params

def forward(params, X):
    z1 = X @ params["W1"] + params["b1"];  a1 = relu(z1)        # (B, 16)
    z2 = a1 @ params["W2"] + params["b2"]; a2 = relu(z2)        # (B, 16)
    z3 = a2 @ params["W3"] + params["b3"]; p = sigmoid(z3)      # (B, 1)
    cache = (X, z1, a1, z2, a2, p)                              # saved for the backward pass
    return p, cache

def bce(p, y, eps=1e-9):
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

params = init([2, 16, 16, 1])
p, _ = forward(params, X)
print(p.shape, bce(p, y))                   # (1000, 1) about 0.7: random guessing

## Backpropagation

### Deriving it, one layer at a time

Check the shapes, which is how you remember these: $a^\top$ is `(16, B)` and $\delta$ is `(B, 1)`, so $a^\top\delta$ is `(16, 1)`, the shape of $W_3$. The gradient of every parameter always has the same shape as the parameter.

In [ ]:
def backward(params, cache, y):
    X, z1, a1, z2, a2, p = cache
    B = len(y)
    dz3 = (p - y) / B                           # (B, 1)
    grads = {"W3": a2.T @ dz3, "b3": dz3.sum(0)}
    da2 = dz3 @ params["W3"].T                  # (B, 16)
    dz2 = da2 * (z2 > 0)                        # through ReLU
    grads["W2"] = a1.T @ dz2
    grads["b2"] = dz2.sum(0)
    da1 = dz2 @ params["W2"].T                  # (B, 16)
    dz1 = da1 * (z1 > 0)
    grads["W1"] = X.T @ dz1
    grads["b1"] = dz1.sum(0)
    return grads

### Trust, but verify

Never trust a hand-written backward pass until it passes a gradient check (lesson 02).

In [ ]:
params = init([2, 16, 16, 1])
p, cache = forward(params, X[:20])
g = backward(params, cache, y[:20])

eps = 1e-5
for k in ["W1", "b2", "W3"]:
    idx = (0, 0) if params[k].ndim == 2 else (0,)
    old = params[k][idx]
    params[k][idx] = old + eps; lp = bce(forward(params, X[:20])[0], y[:20])
    params[k][idx] = old - eps; lm = bce(forward(params, X[:20])[0], y[:20])
    params[k][idx] = old
    print(k, g[k][idx], (lp - lm) / (2 * eps))   # the two columns should agree to ~8 digits

## Training it

### Optimisers: momentum and Adam

where $\hat m$ and $\hat v$ are bias-corrected versions that fix the cold start at zero.

In [ ]:
class Adam:
    def __init__(self, params, lr=1e-2, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}

    def step(self, params, grads):
        self.t += 1
        for k in params:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            m_hat = self.m[k] / (1 - self.b1 ** self.t)
            v_hat = self.v[k] / (1 - self.b2 ** self.t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

### The training loop

In [ ]:
idx = rng.permutation(len(X))
tr, va = idx[:800], idx[800:]

params = init([2, 16, 16, 1])
opt = Adam(params, lr=0.01)
for epoch in range(200):
    perm = rng.permutation(tr)                       # reshuffle every epoch
    for start in range(0, len(perm), 32):
        b = perm[start:start + 32]
        p, cache = forward(params, X[b])             # 1. forward
        grads = backward(params, cache, y[b])        # 2. backward
        opt.step(params, grads)                      # 3. update
    if epoch % 50 == 0:
        p_va, _ = forward(params, X[va])
        print(epoch, round(((p_va > 0.5) == y[va]).mean(), 3))
# validation accuracy reaches about 0.96 to 0.97; logistic regression manages about 0.89

## Making deep networks train

### Hands-on: experiments that build intuition

Using the NumPy network above, run each experiment and write one sentence in your log about what you saw.

1. Remove both ReLUs (make them identity functions). What happens to validation accuracy, and why?
2. Replace He initialisation with `rng.normal(0, 1.0, ...)` and with `rng.normal(0, 0.01, ...)`. Print the loss in the first few epochs.
3. Replace Adam with plain SGD at learning rates 0.01, 0.1 and 1.0.
4. Stretch: add dropout to the hidden layers in training (multiply by a random mask, divide by the keep probability) and check whether the gap between training and validation accuracy shrinks with `noise=0.35`.

#### Solution: what you should find

1. **No activations**: accuracy falls to about 0.89, the same as logistic regression. Without non-linearity the whole network collapses into one linear layer, so it can only draw a straight boundary.
2. **Initialisation**: with standard deviation 1.0, initial activations and the loss are large and training is erratic at first (it may recover on such a small network). With 0.01, the signal nearly vanishes through the layers and the loss sits near 0.69 for a while before learning starts. Deeper networks would fail outright.
3. **SGD**: 0.01 learns slowly, 0.1 works well (about 0.96), and 1.0 may oscillate or diverge. Adam is far less sensitive to the exact learning rate, which is a big reason for its popularity.
4. **Dropout** in the backward pass must apply the same mask to the gradient:

*Shown for reference, not run here:*

```python
keep = 0.9
mask1 = (rng.random(a1.shape) < keep) / keep
a1 = a1 * mask1             # forward, training only
# backward: da1 = da1 * mask1 before multiplying by the ReLU mask
```

On noisier data the train/validation gap usually narrows a little. On a problem this small the effect is modest; on large networks it matters much more.

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/05-neural-networks.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.